# Day 4-4: Grad-CAM & 최종 평가

**강의 시간**: 2시간  
**학습 목표**:
- Grad-CAM으로 모델 해석
- 의료 AI 설명 가능성
- ROC-AUC, PR Curve 분석
- 최종 종합 평가

**사전 요구사항**: Day 4-3 완료  
**Best 모델**: Class Weights (COVID Recall 96.96%)

## 🔧 0. 환경 설정

In [ ]:
# 라이브러리 설치
%pip install -q 'mlflow>=2,<3' dagshub tensorflow opencv-python scikit-learn

print("✅ 라이브러리 설치 완료!")

In [ ]:
# 라이브러리 임포트
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.metrics import (
    classification_report, confusion_matrix,
    balanced_accuracy_score, recall_score, f1_score
)
from sklearn.utils.class_weight import compute_class_weight
import warnings
warnings.filterwarnings('ignore')

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Model
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.applications.resnet50 import preprocess_input
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

import mlflow
import dagshub

np.random.seed(42)
tf.random.set_seed(42)

print(f"✅ TensorFlow {tf.__version__}")

In [ ]:
# 시각화 설정
sns.set_style('whitegrid')

# 1) 폰트 파일 직접 다운로드 (런타임 재시작 불필요)
!wget -q -O NanumGothic.ttf -L "https://fonts.gstatic.com/ea/nanumgothic/v5/NanumGothic-Regular.ttf"

import matplotlib.font_manager as fm

font_path = "NanumGothic.ttf"
fm.fontManager.addfont(font_path)

plt.rcParams['figure.dpi'] = 300
plt.rcParams['savefig.dpi'] = 300

font_prop = fm.FontProperties(fname=font_path)
plt.rcParams["font.family"] = font_prop.get_name()
plt.rcParams["axes.unicode_minus"] = False

🔥 이 부분은 수정이 필요합니다.

**repo_owner**와 **repo_name**을 본인의 Dagshub 정보로 채워 주세요.

In [ ]:
# MLflow 설정
import mlflow
import dagshub

repo_owner = # 🔥 직접 작성이 필요합니다.
repo_name  = # 🔥 직접 작성이 필요합니다.

dagshub.init(repo_owner=repo_owner, repo_name=repo_name, mlflow=True)
mlflow.set_experiment('day4-covid-xray-classification')
print('✅ MLflow 설정 완료!')

## 📂 1. Best 모델 & 데이터 로드

In [ ]:
# MLflow에서 Best model 로드 (Day 4-3 Class Weights)
print("🔍 MLflow에서 Best model 찾는 중...")

experiment = mlflow.get_experiment_by_name('day4-covid-xray-classification')

# Class Weights 모델 찾기
runs = mlflow.search_runs(
    experiment_ids=[experiment.experiment_id],
    filter_string="tags.mlflow.runName = 'ResNet50_ClassWeights'",
    order_by=["start_time DESC"],
    max_results=1
)

if len(runs) == 0:
    print("⚠️ ResNet50_ClassWeights를 찾을 수 없습니다.")
    print("   가장 높은 성능의 모델을 로드합니다...")
    runs = mlflow.search_runs(
        experiment_ids=[experiment.experiment_id],
        filter_string="params.strategy = 'Class Weights'",
        order_by=["metrics.val_accuracy DESC"],
        max_results=1
    )

if len(runs) == 0:
    raise ValueError("MLflow에서 모델을 찾을 수 없습니다! Day 4-3을 먼저 완료하세요.")

best_run = runs.iloc[0]
run_id = best_run['run_id']
run_name = best_run.get('tags.mlflow.runName', 'Unknown')
val_acc = best_run.get('metrics.val_accuracy', 'N/A')

print(f"\n✅ Best model 발견!")
print(f"   Run Name: {run_name}")
print(f"   Run ID: {run_id}")
print(f"   Val Accuracy: {val_acc}")

print(f"\n📥 모델 로드 중...")
model_uri = f"runs:/{run_id}/model"
best_model = mlflow.keras.load_model(model_uri)

print(f"\n✅ 모델 로드 완료!")
print(f"   Strategy: Class Weights")

In [ ]:
import os
from google.colab import userdata

os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_API_TOKEN')
os.environ['KAGGLE_API_TOKEN'] = userdata.get('KAGGLE_API_TOKEN')

try:
    from kaggle.api.kaggle_api_extended import KaggleApi

    api = KaggleApi()
    api.authenticate()

    dataset_id = 'tawsifurrahman/covid19-radiography-database'
    print(f"📥 {dataset_id} 다운로드 시작...")

    api.dataset_download_files(
        dataset_id,
        path='./data',
        unzip=True,
        quiet=False
    )

    print("\n✅ 다운로드 및 압축 해제 완료!")

except Exception as e:
    print(f"\n❌ 오류 발생: {e}")

In [ ]:
# 데이터 로드
data_dir = Path('./data/COVID-19_Radiography_Dataset')
class_names = ['COVID', 'Lung_Opacity', 'Normal', 'Viral Pneumonia']

def get_file_paths_and_labels(data_dir):
    class_to_idx = {name: idx for idx, name in enumerate(class_names)}
    file_paths, labels = [], []

    for class_name in class_names:
        for img_path in (data_dir / class_name / 'images').glob('*.png'):
            file_paths.append(str(img_path))
            labels.append(class_to_idx[class_name])

    return file_paths, labels

file_paths, labels = get_file_paths_and_labels(data_dir)

from sklearn.model_selection import train_test_split
train_paths, val_paths, train_labels, val_labels = train_test_split(
    file_paths, labels, test_size=0.2, stratify=labels, random_state=42
)

print(f"✅ Val: {len(val_paths):,}개")

In [ ]:
# Preprocessing & Dataset
def load_and_preprocess(path, label):
    img = tf.io.read_file(path)
    img = tf.image.decode_png(img, channels=1)
    img = tf.image.resize(img, (224, 224))
    img = tf.image.grayscale_to_rgb(img)
    img = preprocess_input(img)
    return img, label

BATCH_SIZE = 32
val_dataset = tf.data.Dataset.from_tensor_slices((val_paths, val_labels))
val_dataset = val_dataset.map(load_and_preprocess)
val_dataset = val_dataset.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

print("✅ Dataset 생성 완료!")

## 👁️ 2. Grad-CAM 구현

🔥 이 부분을 같이 작성해봅시다.

Grad-CAM의 핵심인 **채널별 중요도(pooled_grads)**, **Weighted Sum**, **ReLU + 정규화**를 완성해 보세요.

```
α^k = Global Average Pooling(∂loss / ∂A^k)
     = tf.reduce_mean(grads, axis=(0, 1, 2))

L = ReLU(Σ α^k × A^k)
  = tf.maximum(heatmap, 0)
```

In [ ]:
def make_gradcam_heatmap(model, img_array, pred_index=None):
    # 1. ResNet50 base model 찾기
    base_model = None
    for layer in model.layers:
        if 'resnet' in layer.name.lower():
            base_model = layer
            break

    if base_model is None:
        raise ValueError("ResNet50 base model을 찾을 수 없습니다!")

    # 2. Grad 모델 구성 (마지막 Conv Layer 출력 + 전체 예측 출력)
    last_conv_layer_name = 'conv5_block3_out'
    grad_model = tf.keras.models.Model(
        inputs=[base_model.input],
        outputs=[base_model.get_layer(last_conv_layer_name).output, base_model.output]
    )

    # 3. Gradient 계산 (GradientTape)
    with tf.GradientTape() as tape:
        last_conv_layer_output, base_output = grad_model(img_array)

        x = base_output
        start_passing = False
        for layer in model.layers:
            if layer == base_model:
                start_passing = True
                continue
            if start_passing:
                x = layer(x)

        predictions = x
        if pred_index is None:
            pred_index = tf.argmax(predictions[0])
        loss = predictions[:, pred_index]

    # 4. ∂loss / ∂conv_outputs
    grads = tape.gradient(loss, last_conv_layer_output)

    # 5. 채널별 중요도 (Global Average Pooling)
    pooled_grads = # 🔥 직접 작성이 필요합니다. (tf.reduce_mean(grads, axis=(0, 1, 2)))

    # 6. Weighted Sum
    last_conv_layer_output = last_conv_layer_output[0]
    heatmap = # 🔥 직접 작성이 필요합니다. (last_conv_layer_output @ pooled_grads[..., tf.newaxis])
    heatmap = tf.squeeze(heatmap)

    # 7. ReLU + 정규화
    heatmap = # 🔥 직접 작성이 필요합니다. (tf.maximum(heatmap, 0) / (tf.math.reduce_max(heatmap) + 1e-10))

    return heatmap.numpy()

In [ ]:
import cv2

In [ ]:
# Overlay 함수
def overlay_heatmap(img, heatmap, alpha=0.4, colormap=cv2.COLORMAP_JET):
    """Heatmap을 원본 이미지에 오버레이"""

    # Heatmap 리사이즈
    heatmap = cv2.resize(heatmap, (img.shape[1], img.shape[0]))

    # Colormap 적용 (파랑→초록→빨강: JET)
    heatmap = np.uint8(255 * heatmap)
    heatmap = cv2.applyColorMap(heatmap, colormap)

    # RGB 변환
    if len(img.shape) == 2:
        img = cv2.cvtColor(np.uint8(255 * img), cv2.COLOR_GRAY2RGB)
    else:
        img = np.uint8(255 * img)

    # Overlay (alpha 블렌딩)
    overlay = cv2.addWeighted(img, 1-alpha, heatmap, alpha, 0)

    return overlay

print("✅ Overlay 함수 정의 완료!")

## 🎨 3. 클래스별 Grad-CAM 시각화

In [ ]:
# 각 클래스별 샘플 시각화
fig, axes = plt.subplots(4, 4, figsize=(16, 16))

for class_idx, class_name in enumerate(class_names):
    # 해당 클래스 샘플 찾기
    class_indices = [i for i, label in enumerate(val_labels) if label == class_idx]
    sample_idx = np.random.choice(class_indices)

    # 이미지 로드
    img_path = val_paths[sample_idx]
    img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
    img = cv2.resize(img, (224, 224))
    img_normalized = img / 255.0

    # Preprocessing for model
    img_rgb = np.stack([img_normalized]*3, axis=-1)
    img_input = preprocess_input(img_rgb[np.newaxis, ...])

    # 예측
    pred = best_model.predict(img_input, verbose=0)
    pred_class = np.argmax(pred[0])
    pred_prob = pred[0][pred_class]

    # Grad-CAM
    heatmap = make_gradcam_heatmap(best_model, img_input, pred_class)
    overlay = overlay_heatmap(img_normalized, heatmap)

    # 시각화
    axes[class_idx, 0].imshow(img, cmap='gray')
    axes[class_idx, 0].set_title(f'Original\n{class_name}', fontweight='bold')
    axes[class_idx, 0].axis('off')

    axes[class_idx, 1].imshow(heatmap, cmap='jet')
    axes[class_idx, 1].set_title('Grad-CAM', fontweight='bold')
    axes[class_idx, 1].axis('off')

    axes[class_idx, 2].imshow(overlay)
    axes[class_idx, 2].set_title('Overlay', fontweight='bold')
    axes[class_idx, 2].axis('off')

    axes[class_idx, 3].bar(class_names, pred[0], color='steelblue')
    axes[class_idx, 3].set_ylim(0, 1)
    axes[class_idx, 3].set_title(f'Pred: {class_names[pred_class]}\n({pred_prob:.2f})',
                                fontweight='bold')
    axes[class_idx, 3].tick_params(axis='x', rotation=45, labelsize=8)
    axes[class_idx, 3].grid(axis='y', alpha=0.3)

plt.suptitle('Grad-CAM 시각화 — 클래스별 샘플',
             fontsize=18, fontweight='bold', y=0.995)
plt.tight_layout()
plt.savefig('gradcam_by_class.png', dpi=150, bbox_inches='tight')
plt.show()

## 🔍 4. 잘못된 예측 분석 (FP/FN)

In [ ]:
# Validation 전체 예측
print("🔮 Validation 예측 중...")

y_true = []
y_pred = []
y_pred_proba = []

for images, labels in val_dataset:
    preds = best_model.predict(images, verbose=0)
    y_true.extend(labels.numpy())
    y_pred.extend(np.argmax(preds, axis=1))
    y_pred_proba.extend(preds)

y_true = np.array(y_true)
y_pred = np.array(y_pred)
y_pred_proba = np.array(y_pred_proba)

print("✅ 예측 완료!")
print(f"   Accuracy: {np.mean(y_true == y_pred):.4f}")

In [ ]:
# 잘못된 예측 찾기
incorrect_indices = np.where(y_true != y_pred)[0]

print(f"\n잘못된 예측: {len(incorrect_indices)}개 ({len(incorrect_indices)/len(y_true)*100:.2f}%)")

# COVID False Negative 찾기
covid_fn = np.where((y_true == 0) & (y_pred != 0))[0]
print(f"COVID False Negative: {len(covid_fn)}개")

# 샘플 분석
if len(covid_fn) > 0:
    fig, axes = plt.subplots(min(3, len(covid_fn)), 4, figsize=(16, 4*min(3, len(covid_fn))))
    if len(covid_fn) == 1:
        axes = axes[np.newaxis, :]

    for i, idx in enumerate(covid_fn[:3]):
        img_path = val_paths[idx]
        img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
        img = cv2.resize(img, (224, 224))
        img_normalized = img / 255.0

        img_rgb = np.stack([img_normalized]*3, axis=-1)
        img_input = preprocess_input(img_rgb[np.newaxis, ...])

        pred = best_model.predict(img_input, verbose=0)
        pred_class = np.argmax(pred[0])

        heatmap = make_gradcam_heatmap(best_model, img_input, pred_class)
        overlay = overlay_heatmap(img_normalized, heatmap)

        axes[i, 0].imshow(img, cmap='gray')
        axes[i, 0].set_title(f'True: COVID', fontweight='bold', color='red')
        axes[i, 0].axis('off')

        axes[i, 1].imshow(heatmap, cmap='jet')
        axes[i, 1].set_title('Grad-CAM', fontweight='bold')
        axes[i, 1].axis('off')

        axes[i, 2].imshow(overlay)
        axes[i, 2].set_title('Overlay', fontweight='bold')
        axes[i, 2].axis('off')

        axes[i, 3].bar(class_names, pred[0], color='coral')
        axes[i, 3].set_ylim(0, 1)
        axes[i, 3].set_title(f'Pred: {class_names[pred_class]}\n(False Negative)',
                            fontweight='bold', color='red')
        axes[i, 3].tick_params(axis='x', rotation=45, labelsize=8)
        axes[i, 3].grid(axis='y', alpha=0.3)

    plt.suptitle('COVID False Negative 분석 — Grad-CAM',
                 fontsize=16, fontweight='bold', y=0.995)
    plt.tight_layout()
    plt.savefig('covid_false_negative_gradcam.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print("\n🎉 COVID False Negative 없음!")

## 📈 5. ROC-AUC 분석

In [ ]:
from sklearn.preprocessing import label_binarize
from sklearn.metrics import roc_curve, auc

In [ ]:
# One-vs-Rest ROC-AUC
y_true_binary = label_binarize(y_true, classes=[0, 1, 2, 3])

plt.figure(figsize=(10, 8))

roc_auc_scores = {}

for i, class_name in enumerate(class_names):
    fpr, tpr, _ = roc_curve(y_true_binary[:, i], y_pred_proba[:, i])
    roc_auc = auc(fpr, tpr)
    roc_auc_scores[class_name] = roc_auc

    plt.plot(fpr, tpr, linewidth=2.5,
            label=f'{class_name} (AUC = {roc_auc:.3f})')

plt.plot([0, 1], [0, 1], 'k--', linewidth=1.5, label='Random')
plt.xlabel('False Positive Rate', fontweight='bold', fontsize=12)
plt.ylabel('True Positive Rate', fontweight='bold', fontsize=12)
plt.title('ROC Curve (One-vs-Rest)', fontweight='bold', fontsize=14)
plt.legend(loc='lower right', fontsize=10)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('roc_curve.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nROC-AUC Scores:")
print("="*40)
for class_name, score in roc_auc_scores.items():
    print(f"{class_name:20s}: {score:.4f}")
print("="*40)
print(f"Macro Average:        {np.mean(list(roc_auc_scores.values())):.4f}")

## 📊 6. Precision-Recall Curve

In [ ]:
from sklearn.metrics import precision_recall_curve, average_precision_score

In [ ]:
# Precision-Recall Curve
plt.figure(figsize=(10, 8))

ap_scores = {}

for i, class_name in enumerate(class_names):
    precision, recall, _ = precision_recall_curve(
        y_true_binary[:, i],
        y_pred_proba[:, i]
    )
    ap = average_precision_score(y_true_binary[:, i], y_pred_proba[:, i])
    ap_scores[class_name] = ap

    plt.plot(recall, precision, linewidth=2.5,
            label=f'{class_name} (AP = {ap:.3f})')

plt.xlabel('Recall', fontweight='bold', fontsize=12)
plt.ylabel('Precision', fontweight='bold', fontsize=12)
plt.title('Precision-Recall Curve', fontweight='bold', fontsize=14)
plt.legend(loc='lower left', fontsize=10)
plt.grid(alpha=0.3)
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.tight_layout()
plt.savefig('precision_recall_curve.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nAverage Precision Scores:")
print("="*40)
for class_name, score in ap_scores.items():
    print(f"{class_name:20s}: {score:.4f}")
print("="*40)
print(f"Mean Average Precision: {np.mean(list(ap_scores.values())):.4f}")

## 🏆 7. 최종 종합 평가

In [ ]:
# Day 4 전체 여정
journey = {
    'Day 4-1\nBaseline': 0.8542,
    'Day 4-2\nTransfer': 0.9067,
    'Day 4-3\nImbalance': 0.9178
}

covid_recall_journey = {
    'Day 4-1\nBaseline': 0.9571,
    'Day 4-2\nTransfer': 0.8893,
    'Day 4-3\nImbalance': 0.9696
}

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

stages = list(journey.keys())
accs = list(journey.values())

axes[0].plot(range(len(stages)), accs, 'bo-', linewidth=3, markersize=12)
axes[0].set_xticks(range(len(stages)))
axes[0].set_xticklabels(stages)
axes[0].set_ylabel('Validation Accuracy', fontweight='bold', fontsize=12)
axes[0].set_title('Overall Accuracy 향상', fontweight='bold', fontsize=14)
axes[0].set_ylim(0.84, 0.93)
axes[0].grid(alpha=0.3)

for i, (stage, acc) in enumerate(zip(stages, accs)):
    axes[0].text(i, acc + 0.002, f'{acc:.4f}', ha='center', fontweight='bold')

recalls = list(covid_recall_journey.values())

axes[1].plot(range(len(stages)), recalls, 'ro-', linewidth=3, markersize=12)
axes[1].set_xticks(range(len(stages)))
axes[1].set_xticklabels(stages)
axes[1].set_ylabel('COVID-19 Recall', fontweight='bold', fontsize=12)
axes[1].set_title('COVID Recall 복원', fontweight='bold', fontsize=14)
axes[1].set_ylim(0.88, 0.98)
axes[1].axhline(0.95, color='green', linestyle='--', linewidth=2, label='목표 (95%)')
axes[1].legend()
axes[1].grid(alpha=0.3)

for i, (stage, recall) in enumerate(zip(stages, recalls)):
    axes[1].text(i, recall + 0.003, f'{recall:.4f}', ha='center', fontweight='bold')

plt.suptitle('Day 4 Performance Journey — COVID-19 분류',
             fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('day4_journey.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# 최종 성능 표
from sklearn.metrics import precision_recall_fscore_support, balanced_accuracy_score

precision, recall, f1, support = precision_recall_fscore_support(
    y_true, y_pred, average=None
)

final_metrics = pd.DataFrame({
    'Class': class_names,
    'Precision': precision,
    'Recall': recall,
    'F1-Score': f1,
    'Support': support
})

print("\n" + "="*80)
print("  최종 성능 — Class Weights 모델")
print("="*80)
print(final_metrics.to_string(index=False))
print("="*80)

overall_acc = np.mean(y_true == y_pred)
balanced_acc = balanced_accuracy_score(y_true, y_pred)
macro_f1 = np.mean(f1)

print(f"\nOverall Accuracy:      {overall_acc:.4f} ({overall_acc*100:.2f}%)")
print(f"Balanced Accuracy:     {balanced_acc:.4f} ({balanced_acc*100:.2f}%)")
print(f"Macro F1-Score:        {macro_f1:.4f}")
print(f"Mean ROC-AUC:          {np.mean(list(roc_auc_scores.values())):.4f}")
print(f"Mean Avg Precision:    {np.mean(list(ap_scores.values())):.4f}")

print("\n" + "="*80)
print("🎉 목표 달성!")
print("  ✅ Overall Accuracy: 91.78% (목표: 85%+)")
print("  ✅ COVID Recall: 96.96% (목표: 95%+)")
print("  ✅ All Classes F1 > 0.88 (목표: 0.85+)")
print("="*80)

## 🧠 8. 핵심 개념 정리

### Day 4 전체 요약

**데이터**:
- COVID-19 Radiography Database
- 21,165개 이미지 (4-class)
- Class Imbalance: 48% vs 6%

**모델**:
- ResNet50 Transfer Learning
- ImageNet Pretrained
- Class Weights로 Imbalance 해결

**최종 성능**:
```
Overall Accuracy:  91.78%
Balanced Accuracy: 92.81%
COVID Recall:      96.96% ✅
Macro F1:          0.9263
Mean ROC-AUC:      0.99+
```

**기술 스택**:
- Transfer Learning (ImageNet → COVID)
- Data Augmentation
- Class Weights
- Grad-CAM (설명 가능성)
- MLflow (실험 관리)

**달성한 것**:
1. ✅ 의료 AI 구현 (91.78% 정확도)
2. ✅ COVID Recall 96.96% (생명 보호)
3. ✅ 균형 잡힌 성능 (모든 클래스 F1 > 0.88)
4. ✅ 설명 가능성 (Grad-CAM)
5. ✅ 임상 적용 가능 수준

---

### 실무 적용

**1차 스크리닝 도구**:
```
대량 X-ray → AI 분석 → 의사 확인 → 최종 진단

장점:
- 빠른 처리 (초당 수백 장)
- 이상 소견 우선순위
- 의사 업무 경감
```

**지속적 개선**:
- 더 많은 데이터 수집
- External Validation
- 실시간 배포 (API)

축하합니다! Day 4 완전 완료! 🎉

## ✅ Day 4-4 완료 체크리스트

- [ ] Grad-CAM 구현
- [ ] 클래스별 Grad-CAM 시각화
- [ ] 잘못된 예측 분석 (FP/FN)
- [ ] ROC-AUC 계산 및 시각화
- [ ] Precision-Recall Curve
- [ ] Day 4 여정 시각화
- [ ] 최종 성능 종합
- [ ] 목표 달성 확인
- [ ] 설명 가능성 확보
- [ ] 임상 적용 가능성 평가

## 🎉 축하합니다!

### 딥러닝 부트캠프 Day 4 완료!

**당신은 이제 할 수 있습니다:**
- ✅ 의료 이미지 분류 AI 구축
- ✅ Transfer Learning 활용
- ✅ Class Imbalance 해결
- ✅ Grad-CAM으로 모델 해석
- ✅ 의료 AI 윤리 고려
- ✅ 실전 배포 준비

**프로젝트 아이디어:**
- 다른 의료 이미지 (MRI, CT)
- Multi-label Classification
- Segmentation (병변 영역 탐지)
- API 서버 구축

수고하셨습니다! 🚀🎉